In [1]:
import pandas as pd
import numpy as np
import os 
import pandas as pd


In [2]:
def get_metrics_binary(df,column_prediction):
    gt = df["ground_truth"].astype("string").str.strip()
    pr = df[column_prediction].astype("string").str.strip()

    # Optional: if you have literal "NaN" strings
    gt = gt.mask(gt.str.lower().eq("nan"), pd.NA)
    
    is_pos_gt = gt.eq("yes")
    is_neg_gt = gt.eq("no") | gt.isna()
    
    is_pos_pr = pr.eq("yes")
    is_neg_pr = pr.eq("no")  # (add | pr.isna() here if you want NA predictions to count as "no")
    
    TP = (is_pos_gt & is_pos_pr).sum()
    FP = (is_neg_gt & is_pos_pr).sum()
    FN = (is_pos_gt & is_neg_pr).sum()
    TN = (is_neg_gt & is_neg_pr).sum()
    
    precision = TP / (TP + FP) if (TP + FP) else float("nan")
    recall    = TP / (TP + FN) if (TP + FN) else float("nan")
    accuracy=(TP+TN) /(TP+TN+FP+FN)
    print("TP, FP, FN, TN:", int(TP), int(FP), int(FN), int(TN))
    print("Precision:", precision)
    print("Recall:", recall)
    print("Accuracy",accuracy)
    return int(TP), int(FP), int(FN), int(TN),precision,recall,accuracy


## Metrics for interaction with child 

In [3]:
folder= "/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/interaction_w_child/"
for root, dirs, files in os.walk(folder):
    for f in files:
        if f == "predictions.csv":
            full_path = os.path.join(root, f)   # <-- rebuild full path
            print("Results for predictions of gesture type located at :",full_path)
            df = pd.read_csv(full_path)
            column_prediction="prediction"
            if "_12_" in full_path:
                lst=df["raw_prediction"].to_list()
                import re
                raw_texts = [re.search(r"raw_text='(.*?)'", s).group(1) for s in lst]
                df["raw_prediction"]=raw_texts
                column_prediction="raw_prediction"
            get_metrics_binary(df,column_prediction)


Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/interaction_w_child/ovis2/prompt1_12_22/predictions.csv
TP, FP, FN, TN: 916 72 1660 731
Precision: 0.9271255060728745
Recall: 0.3555900621118012
Accuracy 0.4874223142941699
Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/interaction_w_child/ovis2/prompt2_01_12/predictions.csv
TP, FP, FN, TN: 391 24 2186 778
Precision: 0.9421686746987952
Recall: 0.15172681412495148
Accuracy 0.3459603432968334
Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/interaction_w_child/qwen25/prompt2_01_12/predictions.csv
TP, FP, FN, TN: 388 24 2189 778
Precision: 0.941747572815534
Recall: 0.15056266977105162
Accuracy 0.34507250665877476
Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/interaction_w_child/qwen25/prompt1

## Metrics for response to name

In [4]:
folder= "/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/response_to_name/"
for root, dirs, files in os.walk(folder):
    for f in files:
        if f == "predictions.csv":
            full_path = os.path.join(root, f)   # <-- rebuild full path
            print("Results for predictions of gesture type located at :",full_path)
            df = pd.read_csv(full_path)
            column_prediction="prediction"
            if "_12_" in full_path:
                lst=df["raw_prediction"].to_list()
                import re
                raw_texts = [re.search(r"raw_text='(.*?)'", s).group(1) for s in lst]
                df["raw_prediction"]=raw_texts
                column_prediction="raw_prediction"
            get_metrics_binary(df,column_prediction)

Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/response_to_name/ovis2/20251226_1225/predictions.csv
TP, FP, FN, TN: 2 31 3 39
Precision: 0.06060606060606061
Recall: 0.4
Accuracy 0.5466666666666666
Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/response_to_name/ovis2/clips/20260112_1924/predictions.csv
TP, FP, FN, TN: 0 0 125 155
Precision: nan
Recall: 0.0
Accuracy 0.5535714285714286
Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/response_to_name/ovis2/clips/20260113_1413/predictions.csv
TP, FP, FN, TN: 0 0 125 155
Precision: nan
Recall: 0.0
Accuracy 0.5535714285714286
Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/response_to_name/qwen25/clips/20260112_1924/predictions.csv
TP, FP, FN, TN: 0 0 125 155
Precision: nan
Recall: 0.0
Accurac

In [ ]:
/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/response_to_name/ovis2/clips/20260112_1924/predictions.csv

In [25]:
df=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/response_to_name/ovis2/clips/20260112_1924/predictions.csv")
len(df[df["ground_truth"]=="no"])/len(df)

0.4479768786127168

In [31]:
len(df[df["ground_truth"]=="inconsistent"])

66

## Metrics for gesture_type

In [5]:
def predict_top1_top2(df1):
    df=df1.copy()
    df=df[~df["ground_truth"].isna()]
    INVALID="INVALID"
    # Split raw_prediction into top1/top2
    top2 = df["raw_prediction"].astype(str).str.split("|", n=1, expand=True)
    df["top1"] = top2[0].fillna("")

    if top2.shape[1]<2:
        
        valid_top1 = df["prediction"].ne(INVALID) & df["prediction"].ne("")

        top1_acc = (df.loc[valid_top1, "prediction"] == df.loc[valid_top1, "ground_truth"]).mean()
        print("Top-1 accuracy:", top1_acc)
        return None

    df["top2"] = top2[1].fillna("")
    
    # Optionally treat invalids as missing predictions
    valid_top1 = df["top1"].ne(INVALID) & df["top1"].ne("")
    valid_top2 = df["top2"].ne(INVALID) & df["top2"].ne("")
    
    # Top-1 accuracy (ignore rows with invalid top1)
    top1_acc = (df.loc[valid_top1, "top1"] == df.loc[valid_top1, "ground_truth"]).mean()
    
    # Top-2 accuracy: ground_truth in {top1, top2} (ignore rows where both are invalid/empty)
    valid_any = valid_top1 | valid_top2
    top2_acc = (
        (df.loc[valid_any, "ground_truth"] == df.loc[valid_any, "top1"]) |
        (df.loc[valid_any, "ground_truth"] == df.loc[valid_any, "top2"])
    ).mean()
    
    print("Top-1 accuracy:", top1_acc)
    print("Top-2 accuracy:", top2_acc)
    return None

In [6]:
import pandas as pd

def confirm_top1_top2(df, pred_col="raw_prediction", gt_col="ground_truth", invalid="INVALID",predfinal="prediction"):
    # Keep NaNs as NaNs (use pandas nullable string)
    gt = df[gt_col]

    parts = df[pred_col].str.split("|", n=1, expand=True)

    df["top1"] = parts[0] if parts.shape[1] >1 else df[predfinal]
    df["top2"] = parts[1] if parts.shape[1] > 1 else pd.Series(pd.NA, index=df.index, dtype="string")
    cols=["top1","top2"]
    df[cols] = df[cols].astype("string").replace("NaN", pd.NA)
    # helper: NaN-aware equality (counts <NA> == <NA> as True)
    def eq_na(a, b, sentinel="__MISSING__"):
        a2 = a.astype("string").fillna(sentinel)
        b2 = b.astype("string").fillna(sentinel)
        return a2.eq(b2)   # always True/False

    valid_top1 = df["top1"].ne(invalid)   # NaNs will stay NaN here, and that's fine
    valid_top2 = df["top2"].ne(invalid)
    # Top-1 accuracy (ignore INVALID top1 rows, but keep NaNs)
    top1_acc = eq_na(df[ "top1"], gt).mean()
    # Top-2 accuracy (ground truth matches top1 OR top2; NaN==NaN counted as match)
    valid_any = valid_top1 | valid_top2
    top2_acc = (
        eq_na(df.loc[valid_any, "top1"], gt.loc[valid_any]) |
        eq_na(df.loc[valid_any, "top2"], gt.loc[valid_any])
    ).mean()

    print("Top-1 accuracy:", float(top1_acc))
    print("Top-2 accuracy:", float(top2_acc))


In [7]:
folder= "/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/gesture_type/"
for root, dirs, files in os.walk(folder):
    for f in files:
        if f == "predictions.csv":
            full_path = os.path.join(root, f)   # <-- rebuild full path
            print("Results for predictions of gesture type located at :",full_path)
            df = pd.read_csv(full_path)
            confirm_top1_top2(df)
            print("Results for predictions of gesture type, focusing on gestures = yes located at :",full_path)
            df = pd.read_csv(full_path)
            predict_top1_top2(df)


Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/gesture_type/ovis2/clips/prompt_2_01_09_v1/predictions.csv
Top-1 accuracy: 0.2535211267605634
Top-2 accuracy: 0.39436619718309857
Results for predictions of gesture type, focusing on gestures = yes located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/gesture_type/ovis2/clips/prompt_2_01_09_v1/predictions.csv
Top-1 accuracy: 0.28346456692913385
Top-2 accuracy: 0.4015748031496063
Results for predictions of gesture type located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/gesture_type/ovis2/clips/prompt_2_01_09_v2/predictions.csv
Top-1 accuracy: 0.29577464788732394
Top-2 accuracy: 0.3732394366197183
Results for predictions of gesture type, focusing on gestures = yes located at : /orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/gesture_type/ovis2/clips/prompt_2_01_09_v2/predictions.csv
Top-1 accuracy: 0.33070866141732286
Top-2 accuracy:

## Metrics for gesture

In [14]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

df = pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/gesture_type/ovis2/20260113_1230/predictions.csv") 
# or use your df

# --- Treat these as "negative"/missing ---
INVALID_STRINGS = {"NaN", "INVALID", "", "None", "nan"}

def is_missing_gt(x) -> bool:
    # ground_truth may be real NaN or the string "NaN"
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s in INVALID_STRINGS

def is_missing_pred_top1(raw_pred) -> bool:
    # raw_prediction like "reach|clap" or "NaN|point"
    s = "" if pd.isna(raw_pred) else str(raw_pred)
    top1 = s.split("|", 1)[0].strip()
    return (top1 in INVALID_STRINGS)

# Ground-truth binary: 1 = Positive (gesture present), 0 = Negative (no gesture / NaN)
y_true = (~df["ground_truth"].apply(is_missing_gt)).astype(int)

# Prediction binary from top1: 1 = predicted a gesture, 0 = predicted NaN/INVALID
y_pred = (~df["raw_prediction"].apply(is_missing_pred_top1)).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)   # precision for Positive class
rec = recall_score(y_true, y_pred, zero_division=0)       # recall for Positive class
cm = confusion_matrix(y_true, y_pred)  # [[TN, FP],[FN, TP]]

print("Confusion matrix [[TN, FP],[FN, TP]]:\n", cm)
print("Accuracy:", acc)
print("Precision (positive=not-NaN):", prec)
print("Recall (positive=not-NaN):", rec)


Confusion matrix [[TN, FP],[FN, TP]]:
 [[   0 1772]
 [   0 1607]]
Accuracy: 0.4755844924533886
Precision (positive=not-NaN): 0.4755844924533886
Recall (positive=not-NaN): 1.0


In [ ]:
df[df["prediction"].isna()]

In [ ]:
df = results_12_22 # or use your df

# --- Treat these as "negative"/missing ---
INVALID_STRINGS = {"NaN", "INVALID", "", "None", "nan"}

def is_missing_gt(x) -> bool:
    # ground_truth may be real NaN or the string "NaN"
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s in INVALID_STRINGS

def is_missing_pred_top1(raw_pred) -> bool:
    # raw_prediction like "reach|clap" or "NaN|point"
    s = "" if pd.isna(raw_pred) else str(raw_pred)
    top1 = s.split("|", 1)[0].strip()
    return (top1 in INVALID_STRINGS)

# Ground-truth binary: 1 = Positive (gesture present), 0 = Negative (no gesture / NaN)
y_true = (~df["ground_truth"].apply(is_missing_gt)).astype(int)

# Prediction binary from top1: 1 = predicted a gesture, 0 = predicted NaN/INVALID
y_pred = (~df["prediction"].apply(is_missing_pred_top1)).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)   # precision for Positive class
rec = recall_score(y_true, y_pred, zero_division=0)       # recall for Positive class
cm = confusion_matrix(y_true, y_pred)  # [[TN, FP],[FN, TP]]

print("Confusion matrix [[TN, FP],[FN, TP]]:\n", cm)
print("Accuracy:", acc)
print("Precision (positive=not-NaN):", prec)
print("Recall (positive=not-NaN):", rec)


In [ ]:
df[df["prediction"].isna()]

In [1]:
import pandas as pd
import numpy as np
import os 